`Двоичное дерево` — иерархическая структура данных, в которой каждый узел имеет не более двух потомков (детей).
 - первый называется родительским узлом, 
 - а дети называются левым и правым наследниками.

В данном испытании мы будем использовать подвид двоичного дерева — `двоичное дерево поиска`. 
- Правильное дерево не содержит повторяющихся ключей
- для каждого узла гарантируется, что в левом поддереве все значения меньше текущего, а в правом — больше.

#### Двоичное дерево поиска
`oop_testing/solution.py`

Реализуйте класс, который представляет собой узел дерева.

Класс должен содержать:

- Атрибут `key` — ключ узла.
- Атрибуты `left` и `right` — ссылки на левого и правого ребёнка соответственно. 
    - если ребёнок в узле отсутствует, `геттер` возвращает `None`.

Метод `insert(key)` — выполняет добавление узла, формируя правильное двоичное дерево.

```python
from solution import Node
tree = Node()
tree.insert(9)
tree.insert(17)
tree.insert(4)
tree.insert(3)
tree.insert(6)
tree.key  # 9
tree.left.key  # 4
tree.right.key  # 17
tree.left.left.key  # 3
tree.left.right.key  # 6
```

---

#### Разбор решения
---

##### 1. Инициализатор

```python
class Node:
    def __init__(self):
        self.key = None
        self.left = None
        self.right = None
```

Здесь узел создаётся «пустым», а ключ записывается позже (в `insert`). Это хорошо ложится на логику: «сначала есть пустой корень, потом в него вставляем значения».

**Плюсы:**

- Естественная реализация для вставки через метод `insert`: можно сделать так, что корневой узел изначально «пустой контейнер», и первая вставка просто заполняет `key`.
- Проще обрабатывать случай «пустого дерева»: дерево — это просто один `Node()` с `key = None`.
- Меньше параметров при создании: `Node()` вместо `Node(key=...)`.

**Минусы:**

- Узел в промежуточном состоянии может быть «неполным» (с `key = None`). Если где‑то случайно использовать такой узел, легко получить логические ошибки.
- Неявно подразумевается, что «узел без ключа — это нормально», а это не всегда интуитивно.
---

##### 2. Метод insert

```python
def insert(self, key):
        if self.key is None:
            self.key = key
            return
        if key == self.key:
            return
        if key < self.key:
            if not self.left:
                self.left = self.__class__()
            target = self.left
        else:
            if not self.right:
                self.right = self.__class__()
            target = self.right
        target.insert(key)
```
**Шаг 1: заполнение пустого узла**

```python
if self.key is None:
    self.key = key
    return
```
Если текущий узел ещё не имеет ключа (то есть это самый первый узел дерева или новый созданный узел), мы просто записываем туда `key` и завершаем работу. Это позволяет начать дерево с «пустого» узла и заполнить его при первой вставке.

**Шаг 2: защита от дубликатов**

```python
if key == self.key:
    return
```

Если ключ уже есть в этом узле, ничего не делаем. В типичном BST дубликаты либо не хранят, либо хранят по какому‑то правилу (например, в правой ветке). Здесь выбран вариант «не вставлять дубликат».
*BST — это Binary Search Tree, то есть бинарное дерево поиска.*

**Шаг 3: выбор направления (влево или вправо)**

В бинарном дереве поиска действует правило:

- все ключи в левом поддереве меньше текущего ключа;
- все ключи в правом поддереве больше текущего ключа.

Поэтому:

```python
if key < self.key:
    ...
else:
    ...
```

*Если идём влево (`key < self.key`)*

```python
if not self.left:
    self.left = self.__class__()
target = self.left
```
- `if not self.left`: проверяет, есть ли уже левый ребёнок. Если его нет (None), создаём новый узел через self.

    - `self.left` — это атрибут узла (в классе `Node`), который хранит ссылку на левый дочерний узел. 
    - Если левого ребёнка нет, там лежит `None`.
    - В `Python` значение `None` считается «ложным» (`falsy`). То есть условие `if self.left:` будет `True`, только если там реально есть объект (узел).
    - Оператор `not` инвертирует значение: `if not self.left` становится `True`, когда `self.left` равно `None` (то есть левого ребёнка нет).
    
-  ` __class__()` (это то же самое, что `Node()`).
- `target = self.left` — запоминаем ссылку на левого ребёнка, чтобы потом вызвать у него `insert`.

*Если идём вправо (`key >= self.key`, но с учётом предыдущего == это строго >)*

```python
else:
    if not self.right:
        self.right = self.__class__()
    target = self.right
```

Аналогично: если правого ребёнка нет, создаём его; затем сохраняем ссылку в target.

**Шаг 4: рекурсивная вставка**

```python
target.insert(key)
```
- Мы передаём задачу вставки дальше — в найденный (или только что созданный) дочерний узел. 
- Так продолжается, пока не дойдём до какого‑то узла, у которого `key is None`, и он станет местом вставки.


##### Пример работы

**Допустим, делаем:**

```python
root = Node()
root.insert(5)
root.insert(3)
root.insert(7)
root.insert(4)
```
**Что происходит:**

- `root.insert(5)`: узел был пустым → `root.key = 5`.
- `root.insert(3)`: 3 < 5, левого ребёнка нет → создаём, вставляем 3 туда.
- `root.insert(7)`: 7 > 5, правого нет → создаём, вставляем 7.
- `root.insert(4)`: 4 < 5 → идём влево к узлу с 3; 4 > 3 → идём вправо, создаём узел и вставляем 4.

**Итоговая структура:**

    корень: 5
        слева: 3
            справа: 4
        справа: 7
---

##### 3. Фраза из условия «Если ребёнок в узле отсутствует, геттер возвращает None».

В этом коде явных геттеров `get_left()`, `get_right()` нет, но логика та же: 
- атрибуты `self.left` и `self.right` сами по себе хранят либо узел `Node`, либо `None`. 
- при обращении  к  `node.left`, и левого ребёнка нет --> `None`. 
- Это и есть тот самый случай `«нет ребёнка → None»`.
`
Если бы были геттеры, они выглядели бы примерно так:

```python
def get_left(self):
    return self.left  # вернёт Node или None
```
---